In [1]:
import wrds
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.regression.rolling import RollingOLS
import matplotlib.pyplot as plt

/Users/ywo/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
db=wrds.Connection()

Enter your WRDS username [ywo]:ywhan
Enter your password:········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


author's data

In [3]:
df2 = pd.read_csv('/Users/ywo/Downloads/Industry Project/datashare (1)/datashare.csv')
df2['DATE']=pd.to_datetime(df2['DATE'], format='%Y%m%d')
# print(df2)

In [ ]:
print(df2[(df2['DATE']<='1987-12-31')&(df2['permno']==10006)])

beta

In [ ]:
# conn = wrds.Connection()

In [ ]:
# import pandas as pd
# import numpy as np
# import datetime as dt
# import wrds
# from dateutil.relativedelta import *
# from pandas.tseries.offsets import *
# import datetime
# import pickle as pkl
# import pyarrow.feather as feather
# import multiprocessing as mp

In [ ]:


# # CRSP Block
# crsp = conn.raw_sql("""
#                       select a.permno, a.date, a.ret, b.rf, b.mktrf, b.smb, b.hml
#                       from crsp.dsf as a
#                       left join ff.factors_daily as b
#                       on a.date=b.date
#                       WHERE a.date >= '1956-08-30' AND a.date <= '1967-12-31'
#                       """)

# # sort variables by permno and date
# crsp = crsp.sort_values(by=['permno', 'date'])

# # change variable format to int
# crsp['permno'] = crsp['permno'].astype(int)

# # Line up date to be end of month
# crsp['date'] = pd.to_datetime(crsp['date'])

# # add delisting return
# dlret = conn.raw_sql("""
#                      select permno, dlret, dlstdt 
#                      from crsp.dsedelist
#                      """)

# dlret.permno = dlret.permno.astype(int)
# dlret['dlstdt'] = pd.to_datetime(dlret['dlstdt'])
# dlret['date'] = dlret['dlstdt']

# # merge delisting return to crsp return
# crsp = pd.merge(crsp, dlret, how='left', on=['permno', 'date'])
# crsp['dlret'] = crsp['dlret'].fillna(0)
# crsp['ret'] = crsp['ret'].fillna(0)
# crsp['retadj'] = (1 + crsp['ret']) * (1 + crsp['dlret']) - 1
# crsp['exret'] = crsp['retadj'] - crsp['rf']



In [ ]:
# find the closest trading day to the end of the month
# crsp['monthend'] = crsp['date'] + MonthEnd(0)
# crsp['date_diff'] = crsp['monthend'] - crsp['date']
# date_temp = crsp.groupby(['permno', 'monthend'])['date_diff'].min()
# date_temp = pd.DataFrame(date_temp)  # convert Series to DataFrame
# date_temp.reset_index(inplace=True)
# date_temp.rename(columns={'date_diff': 'min_diff'}, inplace=True)
# crsp = pd.merge(crsp, date_temp, how='left', on=['permno', 'monthend'])
# crsp['sig'] = np.where(crsp['date_diff'] == crsp['min_diff'], 1, np.nan)

# # label every date of month end
# crsp['month_count'] = crsp[crsp['sig'] == 1].groupby(['permno']).cumcount()

# # label numbers of months for a firm
# month_num = crsp[crsp['sig'] == 1].groupby(['permno'])['month_count'].tail(1)
# month_num = month_num.astype(int)
# month_num = month_num.reset_index(drop=True)

# # mark the number of each month to each day of this month
# crsp['month_count'] = crsp.groupby(['permno'])['month_count'].fillna(method='bfill')

# # crate a firm list
# df_firm = crsp.drop_duplicates(['permno'])
# df_firm = df_firm[['permno']]
# df_firm['permno'] = df_firm['permno'].astype(int)
# df_firm = df_firm.reset_index(drop=True)
# df_firm = df_firm.reset_index()
# df_firm = df_firm.rename(columns={'index': 'count'})
# df_firm['month_num'] = month_num

# ######################
# # Calculate the beta #
# ######################


# def get_beta(df, firm_list):
#     """

#     :param df: stock dataframe
#     :param firm_list: list of firms matching stock dataframe
#     :return: dataframe with variance of residual
#     """
#     for firm, count, prog in zip(firm_list['permno'], firm_list['month_num'], range(firm_list['permno'].count()+1)):
#         prog = prog + 1
#         print('processing permno %s' % firm, '/', 'finished', '%.2f%%' % ((prog/firm_list['permno'].count())*100))
#         for i in range(count + 1):
#             # if you want to change the rolling window, please change here: i - 2 means 3 months is a window.
#             temp = df[(df['permno'] == firm) & (i - 2 <= df['month_count']) & (df['month_count'] <= i)]
#             # if observations in last 3 months are less 21, we drop the rvar of this month
#             if temp['permno'].count() < 21:
#                 pass
#             else:
#                 rolling_window = temp['permno'].count()
#                 index = temp.tail(1).index
#                 X = np.mat(temp[['mktrf']])
#                 Y = np.mat(temp[['exret']])
#                 ones = np.mat(np.ones(rolling_window)).T
#                 M = np.identity(rolling_window) - ones.dot((ones.T.dot(ones)).I).dot(ones.T)
#                 beta = (X.T.dot(M).dot(X)).I.dot((X.T.dot(M).dot(Y)))
#                 df.loc[index, 'beta'] = beta
#     return df


# def sub_df(start, end, step):
#     """

#     :param start: the quantile to start cutting, usually it should be 0
#     :param end: the quantile to end cutting, usually it should be 1
#     :param step: quantile step
#     :return: a dictionary including all the 'firm_list' dataframe and 'stock data' dataframe
#     """
#     # we use dict to store different sub dataframe
#     temp = {}
#     for i, h in zip(np.arange(start, end, step), range(int((end-start)/step))):
#         print('processing splitting dataframe:', round(i, 2), 'to', round(i + step, 2))
#         if i == 0:  # to get the left point
#             temp['firm' + str(h)] = df_firm[df_firm['count'] <= df_firm['count'].quantile(i + step)]
#             temp['crsp' + str(h)] = pd.merge(crsp, temp['firm' + str(h)], how='left',
#                                              on='permno').dropna(subset=['count'])
#         else:
#             temp['firm' + str(h)] = df_firm[(df_firm['count'].quantile(i) < df_firm['count']) & (
#                     df_firm['count'] <= df_firm['count'].quantile(i + step))]
#             temp['crsp' + str(h)] = pd.merge(crsp, temp['firm' + str(h)], how='left',
#                                              on='permno').dropna(subset=['count'])
#     return temp


# def main(start, end, step):
#     """

#     :param start: the quantile to start cutting, usually it should be 0
#     :param end: the quantile to end cutting, usually it should be 1
#     :param step: quantile step
#     :return: a dataframe with calculated variance of residual
#     """
#     df = sub_df(start, end, step)
#     pool = mp.Pool()
#     p_dict = {}
#     for i in range(int((end-start)/step)):
#         p_dict['p' + str(i)] = pool.apply_async(get_beta, (df['crsp%s' % i], df['firm%s' % i],))
#     pool.close()
#     pool.join()
#     result = pd.DataFrame()
#     print('processing pd.concat')
#     for h in range(int((end-start)/step)):
#         result = pd.concat([result, p_dict['p%s' % h].get()])
#     return result


# # calculate variance of residual through rolling window
# # Note: please split dataframe according to your CPU situation. For example, we split dataframe to (1-0)/0.05 = 20 sub
# # dataframes here, so the function will use 20 cores to calculate variance of residual.
# if __name__ == '__main__':
#     crsp = main(0, 1, 0.05)

# # process dataframe
# crsp = crsp.dropna(subset=['beta'])  # drop NA due to rolling
# crsp = crsp.reset_index(drop=True)
# crsp = crsp[['permno', 'date', 'beta']]



In [4]:
query = """
SELECT 
    a.permno, 
    a.date, 
    a.ret, 
    b.vwretd,   
    c.rf        

FROM 
    crsp.msf AS a
LEFT JOIN 
    crsp.msi AS b ON a.date = b.date
LEFT JOIN 
    ff.factors_monthly AS c ON to_char(a.date, 'YYYY-MM') = to_char(c.date, 'YYYY-MM')
WHERE 
    a.date >= '1950-01-31' AND a.date <= '1987-12-31'
    AND a.ret IS NOT NULL
"""

In [5]:
df1 = db.raw_sql(query)

In [6]:
df1 = df1.drop_duplicates(subset=['permno', 'date'], keep='first')
df1 = df1.reset_index(drop=True)
df1['ret'] = pd.to_numeric(df1['ret'], errors='coerce')
df1['rf'] = pd.to_numeric(df1['rf'], errors='coerce')
df1['vwretd'] = pd.to_numeric(df1['vwretd'], errors='coerce')
# extra return
df1['exret'] = df1['ret'] - df1['rf']
df1['exmkt'] = df1['vwretd'] - df1['rf']
df1['exret'] = df1['exret'].astype('float64')
df1['exmkt'] = df1['exmkt'].astype('float64')

In [ ]:
print(df1)

In [7]:
df1 = df1.sort_values(['permno', 'date'])
df1 = df1.reset_index(drop=True)
grouped = df1.groupby('permno')
cov_rm = grouped[['exret', 'exmkt']].rolling(window=36, min_periods=24).cov().xs('exret', level=2)['exmkt']
var_m = grouped['exmkt'].rolling(window=36, min_periods=24).var()

df1['beta'] = (cov_rm / var_m).reset_index(level=0, drop=True)

In [8]:
df1=df1[df1['date']>='1956-12-31']   
df1.reset_index(drop=True)

print(df1)
# df_3_10014=df3[df3['permno']==10014]
# df_3_10014['beta_1']=df_3_10014['beta'].shift(1)   # compare with raw data
# print(df_3_10014)

         permno        date       ret    vwretd      rf     exret     exmkt  \
0         10000  1986-02-28 -0.257143  0.072501  0.0053 -0.262443  0.067201   
1         10000  1986-03-31  0.365385  0.053887   0.006  0.359385  0.047887   
2         10000  1986-04-30 -0.098592 -0.007903  0.0052 -0.103792 -0.013103   
3         10000  1986-05-30 -0.222656  0.050847  0.0049 -0.227556  0.045947   
4         10000  1986-06-30 -0.005025  0.014244  0.0052 -0.010225  0.009044   
...         ...         ...       ...       ...     ...       ...       ...   
1422517   93324  1985-07-31    -0.125 -0.000251  0.0062 -0.131200 -0.006451   
1422518   93324  1985-08-30 -0.142857 -0.004794  0.0055 -0.148357 -0.010294   
1422519   93324  1985-09-30 -0.166667 -0.039826   0.006 -0.172667 -0.045826   
1422520   93324  1985-10-31       0.4  0.044441  0.0065  0.393500  0.037941   
1422521   93324  1985-11-29 -0.142857  0.069228  0.0061 -0.148957  0.063128   

         beta  
0         NaN  
1         NaN  
2  

In [9]:
df_beta=df1[['permno','date','ret','beta']]
df_beta['betasq']=df_beta['beta']**2
df_beta['beta_1']=df_beta.groupby('permno')['beta'].shift(1)
df_beta['betasq_1']=df_beta.groupby('permno')['betasq'].shift(1)
print(df_beta)

         permno        date       ret  beta  betasq  beta_1  betasq_1
0         10000  1986-02-28 -0.257143   NaN     NaN     NaN       NaN
1         10000  1986-03-31  0.365385   NaN     NaN     NaN       NaN
2         10000  1986-04-30 -0.098592   NaN     NaN     NaN       NaN
3         10000  1986-05-30 -0.222656   NaN     NaN     NaN       NaN
4         10000  1986-06-30 -0.005025   NaN     NaN     NaN       NaN
...         ...         ...       ...   ...     ...     ...       ...
1422517   93324  1985-07-31    -0.125   NaN     NaN     NaN       NaN
1422518   93324  1985-08-30 -0.142857   NaN     NaN     NaN       NaN
1422519   93324  1985-09-30 -0.166667   NaN     NaN     NaN       NaN
1422520   93324  1985-10-31       0.4   NaN     NaN     NaN       NaN
1422521   93324  1985-11-29 -0.142857   NaN     NaN     NaN       NaN

[1336511 rows x 7 columns]


/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_11695/109366609.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_beta['betasq']=df_beta['beta']**2
/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_11695/109366609.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_beta['beta_1']=df_beta.groupby('permno')['beta'].shift(1)
/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_11695/109366609.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

beta trend comparision

In [ ]:
# plt.figure(figsize=(10, 6))
# plt.plot(df_3_10014['date'], df_10014['beta'], label='Author Data', color='blue')
# plt.plot(df_3_10014['date'], df_3_10014['beta_1'], label='My Data', color='red')
# plt.xlabel('Date')
# plt.ylabel('Beta')
# plt.title('beta trend comparision')
# plt.legend() 
# plt.show()

Change in 6 month momentum

In [10]:
df_del_mom=df2[(df2['DATE']>='1957-01-31') & (df2['DATE']<='1967-12-31')]
df_del_mom=df_del_mom[['permno','DATE','chmom','mom6m']]
df_del_mom_10006=df_del_mom[df_del_mom['permno']==10014]
print(df_del_mom_10006)

        permno       DATE     chmom     mom6m
1        10014 1957-01-31 -0.275641 -0.115385
1054     10014 1957-02-28 -0.115385 -0.192308
2107     10014 1957-03-29  0.006667 -0.080000
3165     10014 1957-04-30  0.006667 -0.160000
4224     10014 1957-05-31 -0.012857 -0.160000
...        ...        ...       ...       ...
200262   10014 1967-08-31  0.848777  0.459459
202439   10014 1967-09-29  0.512763  0.729730
204623   10014 1967-10-31  0.613825  0.628571
206808   10014 1967-11-30  0.453868  0.648649
208996   10014 1967-12-29  0.415315  0.729730

[132 rows x 4 columns]


In [11]:
query2= """
SELECT 
    permno, 
    date, 
    ret
    
FROM 
    crsp.msf 

WHERE 
    date >= '1956-01-31' AND date <= '1987-12-31'
    AND ret IS NOT NULL
"""
df_cal_mom = db.raw_sql(query2)

In [12]:
df_cal_mom = df_cal_mom.drop_duplicates(subset=['permno', 'date'], keep='first')
df_cal_mom = df_cal_mom.reset_index(drop=True)
df_cal_mom['ret'] = pd.to_numeric(df_cal_mom['ret'], errors='coerce')
df_cal_mom['ret'] = df_cal_mom['ret'].astype('float64')

In [ ]:
# def calculate_mom(data):
#     mom_now_6=(1+data['ret']).shift(1).rolling(window=6).apply(np.prod, raw=True)-1
#     return mom_now_6
# df_cal_mom['mom6m']=df_cal_mom.groupby('permno', group_keys=False).apply(calculate_mom)

In [13]:
def calculate_delta_mom(data):
    mom_now_6=(1+data['ret']).shift(1).rolling(window=6).apply(np.prod, raw=True)-1
    mom_pre_6=(1+data['ret']).shift(7).rolling(window=6).apply(np.prod, raw=True)-1
    return mom_now_6 - mom_pre_6
df_cal_mom['chmom']=df_cal_mom.groupby('permno', group_keys=False).apply(calculate_delta_mom)


/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_11695/1060906053.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cal_mom['chmom']=df_cal_mom.groupby('permno', group_keys=False).apply(calculate_delta_mom)


In [14]:
df_sd=df_cal_mom[(df_cal_mom['date']>='1957-01-31')]
df_sd = df_sd.reset_index(drop=True)
df_sd=df_sd[['permno','date','chmom']]
print(df_sd)

         permno        date     chmom
0         10006  1957-01-31  0.047180
1         10014  1957-01-31 -0.275640
2         10022  1957-01-31 -0.025491
3         10030  1957-01-31  0.018172
4         10057  1957-01-31  0.025786
...         ...         ...       ...
1335453   93201  1987-12-31 -1.649371
1335454   93220  1987-12-31 -0.964423
1335455   93236  1987-12-31 -1.202767
1335456   93252  1987-12-31 -0.219780
1335457   93316  1987-12-31 -0.552381

[1335458 rows x 3 columns]


return volatility

In [15]:
# author's data
df=df2[(df2['DATE']>='1956-10-31') & (df2['DATE']<='1987-12-31')]
df=df[['permno','DATE','retvol']]
print(df)

         permno       DATE    retvol
0         10006 1957-01-31  0.008058
1         10014 1957-01-31  0.033495
2         10022 1957-01-31  0.015589
3         10030 1957-01-31  0.015849
4         10057 1957-01-31  0.019945
...         ...        ...       ...
1336670   93220 1987-12-31  0.037606
1336671   93236 1987-12-31  0.084282
1336672   93252 1987-12-31  0.020063
1336673   93287 1987-12-31  0.063172
1336674   93316 1987-12-31  0.109943

[1336675 rows x 3 columns]


In [16]:
# calculate with wrds
query3= """
SELECT 
    permno, 
    date, 
    ret
    
FROM 
    crsp.dsf 

WHERE 
    date >= '1956-10-31' AND date <= '1987-12-31'
    AND ret IS NOT NULL
"""
df_cal_vol = db.raw_sql(query3)
df_cal_vol['date']=pd.to_datetime(df_cal_vol['date'])
df_cal_vol = df_cal_vol.drop_duplicates(subset=['permno', 'date'], keep='first')
df_cal_vol = df_cal_vol.reset_index(drop=True)
df_cal_vol['ret'] = pd.to_numeric(df_cal_vol['ret'], errors='coerce')
df_cal_vol['ret'] = df_cal_vol['ret'].astype('float64')
df_cal_vol['year_month'] = df_cal_vol['date'].dt.to_period('M')
# print(df_cal_vol)

In [17]:
df_cal_vol = df_cal_vol.sort_values(['permno', 'year_month'])
def calculate_volatility(data):
    if len(data) < 15:
        return np.nan
    return data['ret'].std()
df_cal_vol2 = df_cal_vol.groupby(['permno','year_month'], group_keys=False).apply(calculate_volatility).reset_index()


/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_11695/510702948.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cal_vol2 = df_cal_vol.groupby(['permno','year_month'], group_keys=False).apply(calculate_volatility).reset_index()


In [18]:
df_cal_vol2.columns=['permno','year_month','retvol']
df_cal_vol2['date'] = df_cal_vol2['year_month'].dt.to_timestamp('M')
# print(df_cal_vol2)


In [27]:
df_vol=df_cal_vol2
df_vol['retvol_1']=df_vol.groupby('permno')['retvol'].shift(1)
print(df_vol)

         permno year_month    retvol       date  retvol_1
0         10000    1986-01  0.065278 1986-01-31       NaN
1         10000    1986-02  0.031004 1986-02-28  0.065278
2         10000    1986-03  0.044548 1986-03-31  0.031004
3         10000    1986-04  0.011246 1986-04-30  0.044548
4         10000    1986-05  0.038862 1986-05-31  0.011246
...         ...        ...       ...        ...       ...
1361437   93324    1985-08  0.030457 1985-08-31  0.065452
1361438   93324    1985-09  0.072503 1985-09-30  0.030457
1361439   93324    1985-10  0.145448 1985-10-31  0.072503
1361440   93324    1985-11  0.031944 1985-11-30  0.145448
1361441   93324    1985-12       NaN 1985-12-31  0.031944

[1361442 rows x 5 columns]


In [ ]:
# merged1=pd.merge(df,df_cal_vol,left_on=['permno','DATE'],right_on=['permno','date'],how='right')
# merged1=merged1[['permno','date','year_month','ret']]
# print(merged1)
# merged2=pd.merge(merged1,df_cal_vol2,on=['permno','year_month'],how='left')
# print(merged2)

In [ ]:
# # final calculate volatility dataset
# df_vol=merged2[['permno','date','retvol']]
# df_vol=df_vol[(df_vol['date']>='1956-12-31')]
# df_vol = df_vol.reset_index(drop=True)
# df_vol['retvol_1']=df_vol.groupby('permno')['retvol'].shift(1)
# print(df_vol)


size

In [20]:
query = """
SELECT 
    permno,      
    date,        
     
    (abs(prc) * shrout) / 1000 AS mvel1 
FROM 
    crsp.msf
WHERE 
    date >= '1956-10-31' AND date <= '1987-12-31'
"""

In [21]:
df_size = db.raw_sql(query)
df_size = df_size.drop_duplicates(subset=['permno', 'date'], keep='first')
df_size = df_size.reset_index(drop=True)

In [22]:
df_size=df_size[df_size['date']>='1956-12-31']
df_size = df_size.reset_index(drop=True)
df_size['mvel1_1']=df_size.groupby('permno')['mvel1'].shift(1)
print(df_size)

         permno        date      mvel1    mvel1_1
0         10006  1956-12-31     82.249       <NA>
1         10014  1956-12-31   3.903375       <NA>
2         10022  1956-12-31    9.27325       <NA>
3         10030  1956-12-31  54.465875       <NA>
4         10057  1956-12-31      40.25       <NA>
...         ...         ...        ...        ...
1407465   93220  1987-12-31  109.05375   85.56525
1407466   93236  1987-12-31   8.164625    9.49375
1407467   93252  1987-12-31     8.8275       9.63
1407468   93287  1987-12-31       <NA>  21.858625
1407469   93316  1987-12-31      5.316      5.316

[1407470 rows x 4 columns]


take beta, size, chmom, volatility together

In [31]:
df_beta['date'] = pd.to_datetime(df_beta['date'])
df_size['date'] = pd.to_datetime(df_size['date'])
df_vol['date'] = pd.to_datetime(df_vol['date'])
df_sd['date'] = pd.to_datetime(df_sd['date'])
mergeda=pd.merge(df_beta,df_size,on=['permno','date'],how='right')
mergedb=pd.merge(mergeda,df_vol,on=['permno','date'],how='left')
mergedc=pd.merge(mergedb,df_sd,on=['permno','date'],how='left')
mergedc=mergedc[mergedc['date']>='1957-01-31']
mergedc = mergedc.reset_index(drop=True)
print(mergeda)
print(mergedb)
print(mergedc)

/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_11695/2236687341.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_beta['date'] = pd.to_datetime(df_beta['date'])


         permno       date       ret      beta    betasq    beta_1  betasq_1  \
0         10006 1956-12-31  0.044843  0.901440  0.812594       NaN       NaN   
1         10014 1956-12-31 -0.086957  0.431768  0.186424       NaN       NaN   
2         10022 1956-12-31 -0.060377  0.740043  0.547664       NaN       NaN   
3         10030 1956-12-31  0.044633  0.754648  0.569494       NaN       NaN   
4         10057 1956-12-31  0.086667  1.446529  2.092445       NaN       NaN   
...         ...        ...       ...       ...       ...       ...       ...   
1407465   93220 1987-12-31   0.27451  1.050098  1.102705  0.953567  0.909290   
1407466   93236 1987-12-31     -0.14  2.050259  4.203563  2.197215  4.827756   
1407467   93252 1987-12-31 -0.083333  0.887207  0.787136  0.956305  0.914520   
1407468   93287 1987-12-31      <NA>       NaN       NaN       NaN       NaN   
1407469   93316 1987-12-31       0.0  1.012256  1.024662  1.021879  1.044236   

             mvel1    mvel1_1  
0      

In [32]:
df=mergedc[['permno','date','ret','beta_1','betasq_1','mvel1_1','retvol_1','chmom']]
df.columns=['permno','date','ret','beta','betasq','mvel1','retvol','chmom']
print(df)

         permno       date       ret      beta    betasq      mvel1    retvol  \
0         10006 1957-01-31  0.064378  0.901440  0.812594     82.249  0.008058   
1         10014 1957-01-31  0.095238  0.431768  0.186424   3.903375  0.033495   
2         10022 1957-01-31  0.102041  0.740043  0.547664    9.27325  0.015589   
3         10030 1957-01-31 -0.047091  0.754648  0.569494  54.465875  0.015849   
4         10057 1957-01-31 -0.090062  1.446529  2.092445      40.25  0.019945   
...         ...        ...       ...       ...       ...        ...       ...   
1406401   93220 1987-12-31   0.27451  0.953567  0.909290   85.56525  0.037606   
1406402   93236 1987-12-31     -0.14  2.197215  4.827756    9.49375  0.084282   
1406403   93252 1987-12-31 -0.083333  0.956305  0.914520       9.63  0.020063   
1406404   93287 1987-12-31      <NA>       NaN       NaN  21.858625  0.063171   
1406405   93316 1987-12-31       0.0  1.021879  1.044236      5.316  0.109943   

            chmom  
0      

In [33]:
print(df[df['permno']==10006])

         permno       date       ret      beta    betasq       mvel1  \
0         10006 1957-01-31  0.064378  0.901440  0.812594      82.249   
1066      10006 1957-02-28  0.002016  0.761639  0.580094      87.544   
2131      10006 1957-03-29  0.018405  0.756499  0.572290     86.3085   
3200      10006 1957-04-30 -0.008032  0.788157  0.621192      87.897   
4276      10006 1957-05-31  0.004049  0.781893  0.611356      87.191   
...         ...        ...       ...       ...       ...         ...   
1088655   10006 1984-02-29  0.054247  0.859432  0.738623  382.565625   
1095121   10006 1984-03-30  0.068063  0.799626  0.639402   400.38375   
1101598   10006 1984-04-30  0.031863  0.826172  0.682560     427.635   
1108101   10006 1984-05-31       0.0  0.803114  0.644992  441.260625   
1114637   10006 1984-06-29      <NA>       NaN       NaN  441.260625   

           retvol     chmom  
0        0.008058  0.047180  
1066     0.012694  0.030331  
2131          NaN  0.134574  
3200     0.0100

In [34]:
df.to_csv('/Users/ywo/Downloads/Industry Project/feature construction.csv', index=False)


In [35]:
db.close()